# Protenix Inference (Colab)

This notebook runs Protenix inference from a local sequence file (for example `data/a5b1/sequences/sequences_updated`) and writes a Protenix JSON input automatically.

- Latest known Protenix release at notebook creation: `v1.0.4` (2026-02-09).
- You can let the notebook auto-resolve the latest release at runtime.


In [ ]:
from pathlib import Path
import os

IN_COLAB = "COLAB_GPU" in os.environ
print("IN_COLAB =", IN_COLAB)

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")


IN_COLAB = True
Drive mount skipped: mount failed


In [ ]:
import json
import subprocess
import sys
import urllib.request

USE_LATEST_RELEASE = True
PINNED_TAG = "v1.0.4"  # latest known on 2026-02-13

tag = PINNED_TAG
if USE_LATEST_RELEASE:
    api_url = "https://api.github.com/repos/bytedance/Protenix/releases/latest"
    with urllib.request.urlopen(api_url) as response:
        tag = json.load(response)["tag_name"]

version = tag.lstrip("v")
print(f"Installing Protenix from release tag: {tag}")

install_attempts = [
    [sys.executable, "-m", "pip", "install", "-q", "-U", f"protenix=={version}"],
    [sys.executable, "-m", "pip", "install", "-q", "-U", f"git+https://github.com/bytedance/Protenix.git@{tag}"],
]

last_rc = 1
for cmd in install_attempts:
    print(">", " ".join(cmd))
    last_rc = subprocess.run(cmd).returncode
    if last_rc == 0:
        break

if last_rc != 0:
    raise RuntimeError("Failed to install Protenix from both PyPI and GitHub tag.")

subprocess.run(["protenix", "--help"], check=True)


Installing Protenix from release tag: v1.0.4
> /usr/bin/python3 -m pip install -q -U protenix==1.0.4


CompletedProcess(args=['protenix', '--help'], returncode=0)

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

CANDIDATE_SEQUENCE_FILES = [
    Path("data/a5b1/sequences/sequences_updated"),
    Path("data/sequence updated"),
    Path("/content/Protenix/data/a5b1/sequences/sequences_updated"),
    Path("/content/Protenix/data/sequence updated"),
    Path("/content/drive/MyDrive/colab_cache/Protenix/data/a5b1/sequences/sequences_updated"),
    Path("/content/drive/MyDrive/colab_cache/Protenix/data/sequence updated"),
]

TARGET_MODE = "tagged_heterodimer"  # "tagged_heterodimer", "integrin_ab", "single", "complex"
TARGET_INDEX = 0
JOB_NAME_OVERRIDE = "tagged_integrin_heterodimer"

INTEGRIN_AB_COMPONENTS = [
    "Integrin alpha5-Avi",
    "Integrin beta1-spycatcher",
]

TAGGED_HETERODIMER_COMPONENTS = [
    "Integrin alpha5-Avi",
    "Streptavidin",
    "Integrin beta1-spycatcher",
    "Spytag",
]

WORK_DIR = Path("/content/protenix_colab")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def pick_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    listed = "\n".join(str(p) for p in paths)
    raise FileNotFoundError(f"No sequence file found. Checked:\n{listed}")


def parse_fasta_like(path):
    records = []
    header = None
    seq_parts = []

    with path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if set(line) == {"="}:
                continue

            if line.startswith(">"):
                if header is not None and seq_parts:
                    records.append((header, "".join(seq_parts)))
                header = line[1:].strip() or f"sequence_{len(records) + 1}"
                seq_parts = []
                continue

            cleaned = re.sub(r"[^A-Za-z]", "", line).upper()
            if cleaned:
                seq_parts.append(cleaned)

    if header is not None and seq_parts:
        records.append((header, "".join(seq_parts)))

    if not records:
        raise ValueError(f"No FASTA records found in {path}")

    return records


def slugify(text):
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text or "sequence"


def canonical_name(text):
    text = (
        text.replace("α", "alpha")
        .replace("β", "beta")
        .replace("γ", "gamma")
        .replace("Α", "alpha")
        .replace("Β", "beta")
        .replace("Γ", "gamma")
    )
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^A-Za-z0-9]+", "", text).lower()


def select_by_component_names(records, required_names):
    canonical_to_record = {}
    for name, seq in records:
        key = canonical_name(name)
        if key in canonical_to_record:
            raise ValueError(f"Duplicate canonical sequence name detected: {name}")
        canonical_to_record[key] = (name, seq)

    missing = []
    selected = []
    for required in required_names:
        rec = canonical_to_record.get(canonical_name(required))
        if rec is None:
            missing.append(required)
        else:
            selected.append(rec)

    if missing:
        available = [name for name, _ in records]
        raise ValueError(
            "Missing required components: "
            + ", ".join(missing)
            + "\nAvailable records: "
            + ", ".join(available)
        )

    return selected


sequence_file = pick_existing_path(CANDIDATE_SEQUENCE_FILES)
records = parse_fasta_like(sequence_file)

print(f"Using sequence file: {sequence_file}")
print("Parsed records:")
for i, (name, seq) in enumerate(records):
    print(f"  [{i}] {name} | length={len(seq)}")

if TARGET_MODE not in {"tagged_heterodimer", "integrin_ab", "single", "complex"}:
    raise ValueError("TARGET_MODE must be 'tagged_heterodimer', 'integrin_ab', 'single', or 'complex'.")

if TARGET_MODE == "single":
    if TARGET_INDEX < 0 or TARGET_INDEX >= len(records):
        raise IndexError(f"TARGET_INDEX {TARGET_INDEX} out of range [0, {len(records)-1}]")

    selected_name, selected_seq = records[TARGET_INDEX]
    job_name = JOB_NAME_OVERRIDE or slugify(selected_name)[:80]
    selected_records = [(selected_name, selected_seq)]

elif TARGET_MODE == "complex":
    job_name = JOB_NAME_OVERRIDE or "custom_complex"
    selected_records = records

elif TARGET_MODE == "integrin_ab":
    selected_records = select_by_component_names(records, INTEGRIN_AB_COMPONENTS)
    job_name = JOB_NAME_OVERRIDE or "integrin_alpha5_beta1"
    print("\nSelected integrin alpha/beta components:")
    for i, (name, seq) in enumerate(selected_records):
        print(f"  [{i}] {name} | length={len(seq)}")

else:
    selected_records = select_by_component_names(records, TAGGED_HETERODIMER_COMPONENTS)
    job_name = JOB_NAME_OVERRIDE or "tagged_integrin_heterodimer"
    print("\nSelected tagged heterodimer components:")
    for i, (name, seq) in enumerate(selected_records):
        print(f"  [{i}] {name} | length={len(seq)}")

chain_entries = [
    {
        "proteinChain": {
            "count": 1,
            "sequence": seq,
            "modifications": []
        }
    }
    for _, seq in selected_records
]

payload = [
    {
        "name": job_name,
        "covalent_bonds": [],
        "sequences": chain_entries
    }
]

input_json = WORK_DIR / f"{job_name}.json"
input_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(f"\nWrote Protenix input JSON: {input_json}")
print(input_json.read_text(encoding="utf-8")[:1000])



Using sequence file: /content/drive/MyDrive/colab_cache/Protenix/data/a5b1/sequences/sequences_updated
Parsed records:
  [0] Integrin α5-Avi | length=971
  [1] Streptavidin | length=114
  [2] Integrin β1-spycatcher | length=818
  [3] Spytag | length=16
  [4] DCV | length=120

Selected tagged heterodimer components:
  [0] Integrin α5-Avi | length=971
  [1] Streptavidin | length=114
  [2] Integrin β1-spycatcher | length=818
  [3] Spytag | length=16

Wrote Protenix input JSON: /content/protenix_colab/tagged_integrin_heterodimer.json
[
  {
    "name": "tagged_integrin_heterodimer",
    "covalent_bonds": [],
    "sequences": [
      {
        "proteinChain": {
          "count": 1,
          "sequence": "GSFNLDAEAPAVLSGPPGSFFGFSVEFYRPGTDGVSVLVGAPKANTSQPGVLQGGAVYLCPWGASPTQCTPIEFDSKGSRLLESSLSSSEGEEPVEYKSLQWFGATVRAHGSSILACAPLYSWRTEKEPLSDPVGTCYLSTDNFTRILEYAPCRSDFSWAAGQGYCQGGFSAEFTKTGRVVLGGPGSYFWQGQILSATQEQIAESYYPEYLINLVQGQLQTRQASSIYDDSYLGYSVAVGEFSGDDTEDFVAGVPKGNLTYGYVTILNGSDIRSLYNFSGEQMASYFGYAV

In [ ]:
import subprocess
import torch

MODEL_NAME = "protenix_base_default_v1.0.0"
SEEDS = "101"
USE_MSA = True
USE_TEMPLATE = False
USE_RNA_MSA = False
TRIATT_KERNEL = "torch"
TRIMUL_KERNEL = "torch"

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    DTYPE = "bf16" if major >= 8 else "fp32"
    print(f"GPU: {gpu_name} (cc {major}.{minor}) -> dtype={DTYPE}")
else:
    DTYPE = "fp32"
    print("No CUDA device detected. Inference may be very slow on CPU.")

OUTPUT_DIR = WORK_DIR / f"outputs_{payload[0]['name']}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "protenix", "pred",
    "-i", str(input_json),
    "-o", str(OUTPUT_DIR),
    "-s", SEEDS,
    "-n", MODEL_NAME,
    "--dtype", DTYPE,
    "--use_msa", str(USE_MSA).lower(),
    "--use_template", str(USE_TEMPLATE).lower(),
    "--use_rna_msa", str(USE_RNA_MSA).lower(),
    "--triatt_kernel", TRIATT_KERNEL,
    "--trimul_kernel", TRIMUL_KERNEL,
]

print("Running command:\n", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Inference completed.")


GPU: NVIDIA A100-SXM4-80GB (cc 8.0) -> dtype=bf16
Running command:
 protenix pred -i /content/protenix_colab/tagged_integrin_heterodimer.json -o /content/protenix_colab/outputs_tagged_integrin_heterodimer -s 101 -n protenix_base_default_v1.0.0 --dtype bf16 --use_msa true --use_template false --use_rna_msa false --triatt_kernel torch --trimul_kernel torch
Inference completed.


In [ ]:
all_files = sorted([p for p in OUTPUT_DIR.rglob("*") if p.is_file()])
print(f"Output directory: {OUTPUT_DIR}")
print(f"Total files: {len(all_files)}")

for p in all_files[:200]:
    print(p)

cif_files = sorted(OUTPUT_DIR.rglob("*.cif"))
if cif_files:
    print("\nFirst structure file:")
    print(cif_files[0])
else:
    print("\nNo .cif files found.")


Output directory: /content/protenix_colab/outputs_tagged_integrin_heterodimer
Total files: 28
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/0/non_pairing.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/0/pairing.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/0.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/1/non_pairing.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/1/pairing.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/1.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/2/non_pairing.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/tagged_integrin_heterodimer/msa/2/pairing.a3m
/content/protenix_colab/outputs_tagged_integrin_heterodimer/ta

In [ ]:
import shutil
from pathlib import Path

archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Created local archive:", archive_path)

mydrive_root = Path("/content/drive/MyDrive")
if not mydrive_root.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

drive_results_dir = mydrive_root / "colab_cache" / "Protenix" / "data" / "outputs"
drive_results_dir.mkdir(parents=True, exist_ok=True)

drive_output_dir = drive_results_dir / OUTPUT_DIR.name
if drive_output_dir.exists():
    shutil.rmtree(drive_output_dir)
shutil.copytree(OUTPUT_DIR, drive_output_dir)

drive_archive_path = drive_results_dir / Path(archive_path).name
shutil.copy2(archive_path, drive_archive_path)

print("Copied output directory to:", drive_output_dir)
print("Copied archive to:", drive_archive_path)


Created local archive: /content/protenix_colab/outputs_tagged_integrin_heterodimer.zip
Copied output directory to: /content/drive/MyDrive/colab_cache/Protenix/data/runs/a5b1/protenix/outputs_tagged_integrin_heterodimer
Copied archive to: /content/drive/MyDrive/colab_cache/Protenix/data/runs/a5b1/protenix/outputs_tagged_integrin_heterodimer.zip


# Protenix inference with template

In [4]:
!pip install -q -U protenix gemmi
!apt-get update -y && apt-get install -y kalign

import shutil

KALIGN_BIN = shutil.which("kalign") or shutil.which("kalign3")
if not KALIGN_BIN:
    raise RuntimeError("kalign/kalign3 not found in PATH after install.")
print("Using kalign binary:", KALIGN_BIN)

WORKFLOW_DIR = "/content/drive/MyDrive/colab_cache/Protenix/data/avb3/template_example/workflow_outputs"

!python /content/drive/MyDrive/colab_cache/Protenix/pipelines/protenix-avb3-template/scripts/template_self_sampling_workflow.py \
  --input_pdb /content/drive/MyDrive/colab_cache/Protenix/data/avb3/template_example/seed_090_frame_000.pdb \
  --msa_root /content/drive/MyDrive/colab_cache/afmfold-data/AVB3/seed_090_frame_000/msa \
  --workflow_dir $WORKFLOW_DIR \
  --seeds 101,202,303,404,505,606,707,808,909 \
  --samples_per_seed 5 \
  --kalign_binary_path $KALIGN_BIN \
  --run --run_template_only


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 7,549 B in 1s (6,430 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to pr